# 🚕 03 — Baselines and input-error check (January 2024)

Two questions an interviewer will ask about the fare model, answered on one month of data:

1. **How much does XGBoost beat a simple formula?** Metered fares are mostly distance and time, so a high R² is expected from almost any model. The useful number is the error reduction over a baseline on the same chronological split.
2. **How much worse does it get with request-time inputs?** Training uses each trip's *recorded* distance and duration. A live request only has an *estimate* (Google Routes or Haversine), so this measures how fast the error grows when those inputs are off.

It uses only `yellow_tripdata_2024-01.parquet`, notebook 02's filters and its feature recipe minus weather (`dim_clima` covers just 3 January days, so weather barely varies inside this month). The numbers are indicative; notebook 04 repeats the baseline comparison on the full-year holdout.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import urllib.request
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

if Path('service/ml-inference/data').exists():
    DATA_DIR = Path('service/ml-inference/data')
elif Path('../service/ml-inference/data').exists():
    DATA_DIR = Path('../service/ml-inference/data')
else:
    raise FileNotFoundError("Run this notebook from the project root or the notebooks folder.")

PARQUET = DATA_DIR / "yellow_tripdata_2024-01.parquet"
if not PARQUET.exists():
    print("Downloading January 2024 (~50 MB)...")
    urllib.request.urlretrieve("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet", PARQUET)
print("Using", PARQUET)

Using ../service/ml-inference/data/yellow_tripdata_2024-01.parquet


## 1. Load January with notebook 02's filters and features

In [2]:
cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance', 'fare_amount', 'PULocationID', 'DOLocationID', 'RatecodeID']
df = pd.read_parquet(PARQUET, columns=cols)
df = df[(df.tpep_pickup_datetime >= '2024-01-01') & (df.tpep_pickup_datetime < '2024-02-01')]
df['trip_time'] = (df.tpep_dropoff_datetime - df.tpep_pickup_datetime).dt.total_seconds()
df = df.rename(columns={'trip_distance': 'trip_miles'})
df = df[(df.trip_miles > 0.1) & (df.trip_miles < 100) & (df.fare_amount > 2.5) & (df.fare_amount < 400) & (df.trip_time > 60)]

zones = pd.read_csv(DATA_DIR / "dim_localizacao.csv").rename(columns={'localizacao_id': 'LocationID', 'bairro': 'Borough'})[['LocationID', 'Borough']]
df = df.merge(zones.rename(columns={'LocationID': 'PULocationID', 'Borough': 'PU_Borough'}), on='PULocationID')
df = df.merge(zones.rename(columns={'LocationID': 'DOLocationID', 'Borough': 'DO_Borough'}), on='DOLocationID')
df = df.sort_values('tpep_pickup_datetime').reset_index(drop=True)

df['request_hour'] = df.tpep_pickup_datetime.dt.hour
df['is_weekend'] = (df.tpep_pickup_datetime.dt.dayofweek >= 5).astype('int8')
df['is_airport_trip'] = (df.PULocationID.isin([132, 138, 1]) | df.DOLocationID.isin([132, 138, 1])).astype('int8')
df['implied_speed_mph'] = (df.trip_miles / df.trip_time * 3600).clip(0, 75)
df['route_id'] = df.PULocationID * 1000 + df.DOLocationID
df = pd.get_dummies(df, columns=['PU_Borough', 'DO_Borough'], dtype='uint8')

cut = int(len(df) * 0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()
print(f"Trips after filters: {len(df):,} | train {len(train):,} | test {len(test):,}")
print(f"Test window: {test.tpep_pickup_datetime.min():%Y-%m-%d} to {test.tpep_pickup_datetime.max():%Y-%m-%d}")

Trips after filters: 2,828,853 | train 2,263,082 | test 565,771
Test window: 2024-01-26 to 2024-01-31


## 2. Baselines on the chronological test split

In [3]:
y = test.fare_amount.values
results = []

def score(name, pred):
    results.append({'Method': name,
                    'MAE ($)': round(mean_absolute_error(y, pred), 2),
                    'MAPE (%)': round(100 * mean_absolute_percentage_error(y, pred), 2),
                    'R²': round(r2_score(y, pred), 3)})

minutes = test.trip_time / 60
old_meter = 3.0 + 3.5 * test.trip_miles + np.where(test.implied_speed_mph < 12, 0.7 * minutes, 0)
score('Meter fallback as first written (distance + full time when < 12 mph)', np.maximum(3.0, old_meter))

new_meter = 3.0 + 0.7 * np.maximum(5 * test.trip_miles, minutes)
score('Meter fallback, corrected: 3 + 0.70 × max(5 × miles, minutes)', np.maximum(3.0, new_meter))

lin_cols = ['trip_miles', 'trip_time', 'is_airport_trip']
lin = LinearRegression().fit(train[lin_cols], train.fare_amount)
score('Linear regression (miles, seconds, airport flag)', lin.predict(test[lin_cols]))
pd.DataFrame(results)

,Method,MAE ($),MAPE (%),R²
0,Meter fallback as first written (distance + fu...,5.18,33.88,-1.468
1,"Meter fallback, corrected: 3 + 0.70 × max(5 × ...",2.63,14.33,-1.281
2,"Linear regression (miles, seconds, airport flag)",2.46,17.76,0.923


## 3. XGBoost with notebook 02's recipe (no weather)

Same Bayesian-smoothed route encoding (m = 50, fit on the training split) and notebook 02's tuned hyperparameters.

In [4]:
global_mean = train.fare_amount.mean()
stats = train.groupby('route_id').fare_amount.agg(['mean', 'count'])
stats['smoothed'] = (stats['count'] * stats['mean'] + 50 * global_mean) / (stats['count'] + 50)
train['historical_route_fare'] = train.route_id.map(stats.smoothed)
test['historical_route_fare'] = test.route_id.map(stats.smoothed).fillna(global_mean)

borough_cols = [c for c in df.columns if c.startswith(('PU_Borough_', 'DO_Borough_'))]
features = ['trip_miles', 'trip_time', 'request_hour', 'is_weekend', 'is_airport_trip', 'implied_speed_mph'] + borough_cols + ['historical_route_fare']

model = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.9, colsample_bytree=0.8,
                         tree_method='hist', n_jobs=-1, random_state=42)
model.fit(train[features], train.fare_amount)
xgb_pred = model.predict(test[features])
score('XGBoost, notebook 02 recipe (recorded distance and time)', xgb_pred)
pd.DataFrame(results)

,Method,MAE ($),MAPE (%),R²
0,Meter fallback as first written (distance + fu...,5.18,33.88,-1.468
1,"Meter fallback, corrected: 3 + 0.70 × max(5 × ...",2.63,14.33,-1.281
2,"Linear regression (miles, seconds, airport flag)",2.46,17.76,0.923
3,"XGBoost, notebook 02 recipe (recorded distance...",0.76,4.49,0.972


## 4. Request-time inputs: what if distance and duration are estimates?

Live requests carry estimated distance and duration, not the recorded ones. Here each test trip's inputs are perturbed by a uniform error (time ±15%, distance ±10%, then a wider ±25% / ±20%) and the same model re-scored. Speed is recomputed from the perturbed inputs.

In [5]:
rng = np.random.default_rng(0)
for time_err, dist_err in [(0.15, 0.10), (0.25, 0.20)]:
    noisy = test.copy()
    noisy['trip_time'] = test.trip_time * rng.uniform(1 - time_err, 1 + time_err, len(test))
    noisy['trip_miles'] = test.trip_miles * rng.uniform(1 - dist_err, 1 + dist_err, len(test))
    noisy['implied_speed_mph'] = (noisy.trip_miles / noisy.trip_time * 3600).clip(0, 75)
    score(f'XGBoost with ±{time_err:.0%} time and ±{dist_err:.0%} distance error', model.predict(noisy[features]))

summary = pd.DataFrame(results)
summary

,Method,MAE ($),MAPE (%),R²
0,Meter fallback as first written (distance + fu...,5.18,33.88,-1.468
1,"Meter fallback, corrected: 3 + 0.70 × max(5 × ...",2.63,14.33,-1.281
2,"Linear regression (miles, seconds, airport flag)",2.46,17.76,0.923
3,"XGBoost, notebook 02 recipe (recorded distance...",0.76,4.49,0.972
4,XGBoost with ±15% time and ±10% distance error,1.06,6.24,0.968
5,XGBoost with ±25% time and ±20% distance error,1.45,8.49,0.959


## 5. Summary

In [6]:
lin_mae = summary.loc[summary.Method.str.startswith('Linear'), 'MAE ($)'].item()
xgb_mae = summary.loc[summary.Method.str.startswith('XGBoost, notebook 02'), 'MAE ($)'].item()
print(f"XGBoost cuts MAE by {1 - xgb_mae / lin_mae:.0%} versus the linear baseline (${lin_mae:.2f} -> ${xgb_mae:.2f}).")
imp = pd.Series(model.feature_importances_, features).sort_values(ascending=False).head(5)
print("Top features by gain:", ", ".join(f"{k} {v:.2f}" for k, v in imp.items()))

XGBoost cuts MAE by 69% versus the linear baseline ($2.46 -> $0.76).
Top features by gain: trip_miles 0.59, historical_route_fare 0.16, trip_time 0.10, DO_Borough_EWR 0.06, PU_Borough_Queens 0.01
